## Lab 6: Classification fine-tuning
**Group:** Phillip Graf, Konstantin Schmidt, Fabian Holländer

In this lab, we import our classification dataset, perform necessary preprocessing steps and the train the OpenAI Gpt2 model, such that it can label reciepes into 9 different categories. The cool thing is that multilabeling is possible. That means if a reciepe is a bakery and vegetable the model classify it as it is.

The 9 different categories are:
.  
.  
TODO Beschreibung vollenden...  
.  
.  



For learning and testing purposes we import a reduced classification dataset of 100.000 annotated reciepes.
https://syncandshare.lrz.de/dl/fiWvy4z3fJ2sRSLMTdQiay/reduced_classification_dataset_100k.csv

*Optional* the full dataset containing 2 million annotated reciepes and can be used as input for larger-scale training.
https://syncandshare.lrz.de/dl/fiGzZTKfHqLLLbK2e6Uop9/full_classification_dataset.csv

## Importieren notwendiger Python Bibliotheken

In [84]:
import pandas as pd
import os
import re
import tiktoken
import torch
from torch.utils.data import Dataset, DataLoader
from importlib.metadata import version
from utils.gpt_download import download_and_load_gpt2
from utils.gpt import GPTModel, load_weights_into_gpt
from utils.gpt import (
    generate_text_simple,
    text_to_token_ids,
    token_ids_to_text
)
torch.manual_seed(123)

pkgs = ["matplotlib",  # Plotting library
        "numpy",       # PyTorch & TensorFlow dependency
        "tiktoken",    # Tokenizer
        "torch",       # Deep learning library
        "tensorflow",  # For OpenAI's pretrained weights
        "pandas"       # Dataset loading
       ]
for p in pkgs:
    print(f"{p} version: {version(p)}")

tokenizer = tiktoken.get_encoding("gpt2")

matplotlib version: 3.10.8
numpy version: 2.0.2
tiktoken version: 0.9.0
torch version: 2.10.0
tensorflow version: 2.21.0
pandas version: 2.3.3


## Laden und vorbereiten des Datensatzes

Der Datensatz hat eigentlich 5 Spalten... wir bereiten diese so auf, dass wir nur noch ein Text haben und 2 Classifzierungsspalten. Da im späteren Verlauf ein LLM auch das Gesamte Rezept als Input nimmt und nicht aufgeteilt in verschiedene Spalten.

Wir lassen uns ausgeben welche Label wir haben und wieviele Zeilen pro Label und bringen den Datensatz danach ins Gleichgewicht, damit ein label, dass Training nicht verzerrt oder ein Label nicht untereprsäentiert ist. Deshalb undersampeln wir hier.

Inklusive auf splitten in Traingings und Validierungs und test datensatz

### Design-Entscheidung: Textstrukturierung & Tokenisierung

**Überlegung:**
Ursprünglich wurde in Erwägung gezogen, die Abschnitte der Rezepte (Titel, Zutaten, Anleitung) über explizite Steuerungstokens (*Special Control Tokens* wie `<|startofingredients|>`) zu trennen. Dies ist architektonisch sauber, da das Modell so die semantischen Grenzen im Text exakt erlernen kann.

**Gewählte Lösung & Begründung:**
Wir haben uns letztlich gegen eigene Special Tokens und **für die Strukturierung mittels klassischer Zeilenumbrüche (`\n`)** entschieden. 

* **Problem bei Custom Tokens:** Da wir ein **vortrainiertes GPT-2-Modell** nutzen, würde das Hinzufügen neuer Tokens das originale Vokabular verändern. Die Gewichte des Embedding-Layers würden nicht mehr passen (`Shape Mismatch`), und das Modell müsste für die neuen Tokens komplett neue Vektoren von null auf lernen.
* **Vorteil von `\n`:** GPT-2 wurde auf riesigen Textmengen trainiert und hat bereits verinnerlicht, dass Zeilenumbrüche (`\n`) gefolgt von Schlüsselwörtern wie `Ingredients:` oder `Directions:` Abschnitte strukturieren. Wir nutzen dieses vortrainierte Wissen (*Prior Knowledge*) aus, um das Fine-Tuning stabiler und effizienter zu gestalten.

In [85]:
# Reduced dataset with 100k rows for testing
cloud_url = "https://syncandshare.lrz.de/dl/fiWvy4z3fJ2sRSLMTdQiay/reduced_classification_dataset_100k.csv"
# Uncomment the following line to use the full annotated dataset
# cloud_url = "https://syncandshare.lrz.de/dl/fiGzZTKfHqLLLbK2e6Uop9/full_classification_dataset.csv"

try:
    print("Datensatz aus der Cloud laden...")
    df = pd.read_csv(cloud_url)
    if "Unnamed: 0" in df.columns:
        df = df.drop(columns=["Unnamed: 0"])
    print("Datensatz erfolgreich geladen!\n")
except Exception as e:
    print(f"Ein Fehler ist aufgetreten: {e}")

def combine_recipe_features(row):
    title = str(row['title']).strip()
    ingredients = str(row['NER']).replace('[', '').replace(']', '').replace("'", "").replace('"', '').strip()
    directions = str(row['directions']).replace('[', '').replace(']', '').replace("'", "").replace('"', '').strip()
    return f"Recipe: {title}\nIngredients: {ingredients}\nDirections: {directions}"

print("Bereite Datensatz für das Classification-Fine-Tuning vor...")

df['full_recipe'] = df.apply(combine_recipe_features, axis=1)
df = df[['full_recipe', 'genre', 'label']]

print("Aufbereitung erfolgreich abgeschlossen!\n")
df

Datensatz aus der Cloud laden...
Datensatz erfolgreich geladen!

Bereite Datensatz für das Classification-Fine-Tuning vor...
Aufbereitung erfolgreich abgeschlossen!



,full_recipe,genre,label
0,Recipe: Reeses Cups(Candy)\nIngredients: peanu...,drinks,2
1,Recipe: Rhubarb Coffee Cake\nIngredients: suga...,drinks,2
2,Recipe: Quick Barbecue Wings\nIngredients: chi...,nonveg,3
3,Recipe: Chocolate Frango Mints\nIngredients: c...,drinks,2
4,Recipe: Corral Barbecued Beef Steak Strips\nIn...,drinks,2
...,...,...,...
99995,Recipe: Basic Marinated And Baked Tofu\nIngred...,nonveg,3
99996,Recipe: Bouranee Baunjan - Afghan Eggplant Wit...,sides,8
99997,Recipe: Ginger-Soy-Lime Marinated Shrimp\nIngr...,drinks,2
99998,Recipe: Coffee Can Pumpkin Bread\nIngredients:...,cereal,6


In [86]:
genre_mapping = dict(df[['label', 'genre']].drop_duplicates().values)

In [87]:
print(genre_mapping)

{2: 'drinks', 3: 'nonveg', 4: 'vegetables', 1: 'bakery', 6: 'cereal', 5: 'fastfood', 7: 'meal', 9: 'fusion', 8: 'sides'}


In [88]:
# Anzeige der Anzahl der Rezepte pro Genre und Label
summary_df = df.groupby(['label', 'genre']).size().reset_index(name='count').sort_values('label')
print(summary_df.to_string(index=False))

 label      genre  count
     1     bakery   4112
     2     drinks  21293
     3     nonveg  14714
     4 vegetables  20200
     5   fastfood   6124
     6     cereal  16803
     7       meal   1687
     8      sides  11832
     9     fusion   3235


In [89]:
# Rezeptlänge in Tokens berechnen und filtern auf 1024, da in einem späteren Schritt das GPT-2 Modell eine maximale Kontextlänge von 1024 Tokens hat
df["token_count"] = [len(tokenizer.encode(text)) for text in df["full_recipe"]]
filtered_df = df[df["token_count"] <= 1024].drop(columns=["token_count"])

In [90]:
# Anzeige der Anzahl der Rezepte pro Genre und Label
summary_filtered_df = filtered_df.groupby(['label', 'genre']).size().reset_index(name='count').sort_values('label')
print(summary_filtered_df.to_string(index=False))

 label      genre  count
     1     bakery   4105
     2     drinks  21284
     3     nonveg  14707
     4 vegetables  20196
     5   fastfood   6118
     6     cereal  16789
     7       meal   1685
     8      sides  11826
     9     fusion   3230


In [91]:
# Undersampling um ein gleiches Verhältnis der Klassen zu erreichen
def create_balanced_dataset(df):
    min_size = df['label'].value_counts().min()
    
    subsets = []
    for label in df['label'].unique():
        label_subset = df[df['label'] == label].sample(min_size, random_state=123)
        subsets.append(label_subset)
        
    balanced_df = pd.concat(subsets).reset_index(drop=True)
    return balanced_df

balanced_df = create_balanced_dataset(filtered_df)
print(balanced_df.groupby(['label', 'genre']).size().reset_index(name='count').to_string(index=False))

 label      genre  count
     1     bakery   1685
     2     drinks   1685
     3     nonveg   1685
     4 vegetables   1685
     5   fastfood   1685
     6     cereal   1685
     7       meal   1685
     8      sides   1685
     9     fusion   1685


In [92]:
# Aufteilen in Trainings-, Validierungs- und Testset
def random_split(df, train_frac, validation_frac):
    df = df.sample(frac=1, random_state=123).reset_index(drop=True)

    train_end = int(len(df) * train_frac)
    validation_end = train_end + int(len(df) * validation_frac)

    train_df = df[:train_end]
    validation_df = df[train_end:validation_end]
    test_df = df[validation_end:]

    return train_df, validation_df, test_df

train_df, validation_df, test_df = random_split(balanced_df, 0.7, 0.1)

os.makedirs("classification", exist_ok=True)

train_df.to_csv("classification/train.csv", index=None)
validation_df.to_csv("classification/validation.csv", index=None)
test_df.to_csv("classification/test.csv", index=None)

## Erstellen eines spezifischen Dataloader

Da die Rezepte unterschiedlich lang sind, müssen wir sie für das Batch-Training vereinheitlichen. Dafür füllen wie allte Texte auf die länge des längsten Textes auf (Padding). Als Padding-Token nutzen wir das <|endoftext|> Token.

Training, Validierung und Testdatensatz padden.
- Note that validation and test set samples that are longer than the longest training example are being truncated via `encoded_text[:self.max_length]` in the `SpamDataset` code
- This behavior is entirely optional, and it would also work well if we set `max_length=None` in both the validation and test set cases

In [93]:
class RecipeDataset(Dataset):
    def __init__(self, csv_file, tokenizer, max_length=None, pad_token_id=50256):
        self.data = pd.read_csv(csv_file)

        self.encoded_texts = [
            tokenizer.encode(text) for text in self.data["full_recipe"]
        ]

        if max_length is None:
            self.max_length = self._longest_encoded_length()
        else:
            self.max_length = max_length
            self.encoded_texts = [
                encoded_text[:self.max_length]
                for encoded_text in self.encoded_texts
            ]

        # Pad sequences to the longest sequence
        self.encoded_texts = [
            encoded_text + [pad_token_id] * (self.max_length - len(encoded_text))
            for encoded_text in self.encoded_texts
        ]

    def __getitem__(self, index):
        encoded = self.encoded_texts[index]
        label = self.data.iloc[index]["label"]

        # Um eins verschoben, weil die IDs in genre_mapping bei 1 beginnen, die Modellvorhersage aber bei 0
        shifted_label = int(label) - 1

        return (
            torch.tensor(encoded, dtype=torch.long),
            torch.tensor(shifted_label, dtype=torch.long)
        )

    def __len__(self):
        return len(self.data)

    def _longest_encoded_length(self):
        max_length = 0
        for encoded_text in self.encoded_texts:
            encoded_length = len(encoded_text)
            if encoded_length > max_length:
                max_length = encoded_length
        return max_length

    def _shortest_encoded_length(self):
        if not self.encoded_texts:
            return 0
        min_length = float('inf')
        for encoded_text in self.encoded_texts:
            encoded_length = len(encoded_text)
            if encoded_length < min_length:
                min_length = encoded_length
        return min_length

In [94]:
files = {
    "Train": "classification/train.csv",
    "Validation": "classification/validation.csv",
    "Test": "classification/test.csv"
}

for name, path in files.items():
    df = pd.read_csv(path)
    lengths = [len(tokenizer.encode(text)) for text in df["full_recipe"]]
    
    # Text vorab zusammenbauen
    min_str = f"{min(lengths)} Token"
    max_str = f"{max(lengths)} Token"
    
    # Jetzt den kompletten Text blockweise ausrichten
    print(f"{name:<10} | Kürzestes Rezept: {min_str:<4} | Längstes Rezept: {max_str}")

Train      | Kürzestes Rezept: 22 Token | Längstes Rezept: 982 Token
Validation | Kürzestes Rezept: 28 Token | Längstes Rezept: 833 Token
Test       | Kürzestes Rezept: 30 Token | Längstes Rezept: 879 Token


In [95]:
train_dataset = RecipeDataset(
    csv_file="classification/train.csv",
    max_length=None,
    tokenizer=tokenizer
)

print(train_dataset.max_length)

# Schneide die Texte im Validierungs- und Testset auf die maximale Länge des Trainingssets zu
val_dataset = RecipeDataset(
    csv_file="classification/validation.csv",
    max_length=train_dataset.max_length,
    tokenizer=tokenizer
)

print(val_dataset.max_length)

test_dataset = RecipeDataset(
    csv_file="classification/test.csv",
    max_length=train_dataset.max_length,
    tokenizer=tokenizer
)

print(test_dataset.max_length)

982
982
982


In [96]:
num_workers = 0
batch_size = 8

train_loader = DataLoader(
    dataset=train_dataset,
    batch_size=batch_size,
    shuffle=True,
    num_workers=num_workers,
    drop_last=True,
)

val_loader = DataLoader(
    dataset=val_dataset,
    batch_size=batch_size,
    num_workers=num_workers,
    drop_last=False,
)

test_loader = DataLoader(
    dataset=test_dataset,
    batch_size=batch_size,
    num_workers=num_workers,
    drop_last=False,
)

In [97]:
# verifiziere die Daten in einem Batch
print("Train loader:")
for input_batch, target_batch in train_loader:
    pass

print("Input batch dimensions:", input_batch.shape)
print("Label batch dimensions", target_batch.shape)

Train loader:
Input batch dimensions: torch.Size([8, 982])
Label batch dimensions torch.Size([8])


In [98]:
# Anzahl der Batches in jedem Loader
print(f"{len(train_loader)} training batches")
print(f"{len(val_loader)} validation batches")
print(f"{len(test_loader)} test batches")

1326 training batches
190 validation batches
380 test batches


## Load pre trained model

In [99]:
CHOOSE_MODEL = "gpt2-small (124M)"
INPUT_PROMPT = "Every effort moves"

BASE_CONFIG = {
    "vocab_size": 50257,     # Vocabulary size
    "context_length": 1024,  # Context length !!!!
    "drop_rate": 0.0,        # Dropout rate
    "qkv_bias": True         # Query-key-value bias
}

model_configs = {
    "gpt2-small (124M)": {"emb_dim": 768, "n_layers": 12, "n_heads": 12},
}

BASE_CONFIG.update(model_configs[CHOOSE_MODEL])

assert train_dataset.max_length <= BASE_CONFIG["context_length"], (
    f"Dataset length {train_dataset.max_length} exceeds model's context "
    f"length {BASE_CONFIG['context_length']}. Reinitialize data sets with "
    f"`max_length={BASE_CONFIG['context_length']}`"
)

In [100]:
model_size = CHOOSE_MODEL.split(" ")[-1].lstrip("(").rstrip(")")
settings, params = download_and_load_gpt2(model_size=model_size, models_dir="gpt2")

model = GPTModel(BASE_CONFIG)
load_weights_into_gpt(model, params)
model.eval();

File already exists and is up-to-date: gpt2\124M\checkpoint
File already exists and is up-to-date: gpt2\124M\encoder.json
File already exists and is up-to-date: gpt2\124M\hparams.json
File already exists and is up-to-date: gpt2\124M\model.ckpt.data-00000-of-00001
File already exists and is up-to-date: gpt2\124M\model.ckpt.index
File already exists and is up-to-date: gpt2\124M\model.ckpt.meta
File already exists and is up-to-date: gpt2\124M\vocab.bpe


In [101]:
text_1 = "Recepie: Jewell Ball'S Chicken"

token_ids = generate_text_simple(
    model=model,
    idx=text_to_token_ids(text_1, tokenizer),
    max_new_tokens=15,
    context_size=BASE_CONFIG["context_length"]
)

print(token_ids_to_text(token_ids, tokenizer))

Recepie: Jewell Ball'S Chicken

[00:00:00]EMOTE: *no key*/(


In [102]:
text_2 = (
    "What would you label the following recipe? Answer with 'drinks', 'nonveg', 'vegetables', 'fastfood', 'cereal', 'meal', 'sides', or 'fusion':"
    " Recipe: Meyer Lemon Marmalade\nIngredients: lemons, sugar\nDirections: Rinse the lemons and pat dry. Halve the lemons crosswise and juice them, reserving the juice. Using a spoon, scrape the pulp and seeds from the halves. Using a sharp knife, slice the peels 1/8 inch thick., In a large, heavy saucepan, cover the strips with 8 cups of cold water and bring to a boil; boil for 1 minute. Drain the strips and rinse under cold running water. Blanch two more times; the final time, drain the strips but do not rinse them., Return the strips to the saucepan. Add the reserved juice and the sugar. Simmer over moderate heat, stirring to dissolve the sugar, then skimming any foam, until the marmalade sets, about 30 minutes., Spoon the marmalade into 5 hot 1/2-pint canning jars, leaving 1/4 inch of space at the top, and close with the lids and rings. To process, boil the jars for 15 minutes in water to cover. Let stand at room temperature for 2 days before serving., MAKE AHEAD The processed marmalade can be stored in a cool, dark place for up to 1 year. Refrigerate after opening."
)

token_ids = generate_text_simple(
    model=model,
    idx=text_to_token_ids(text_2, tokenizer),
    max_new_tokens=23,
    context_size=BASE_CONFIG["context_length"]
)

print(token_ids_to_text(token_ids, tokenizer))

What would you label the following recipe? Answer with 'drinks', 'nonveg', 'vegetables', 'fastfood', 'cereal', 'meal', 'sides', or 'fusion': Recipe: Meyer Lemon Marmalade
Ingredients: lemons, sugar
Directions: Rinse the lemons and pat dry. Halve the lemons crosswise and juice them, reserving the juice. Using a spoon, scrape the pulp and seeds from the halves. Using a sharp knife, slice the peels 1/8 inch thick., In a large, heavy saucepan, cover the strips with 8 cups of cold water and bring to a boil; boil for 1 minute. Drain the strips and rinse under cold running water. Blanch two more times; the final time, drain the strips but do not rinse them., Return the strips to the saucepan. Add the reserved juice and the sugar. Simmer over moderate heat, stirring to dissolve the sugar, then skimming any foam, until the marmalade sets, about 30 minutes., Spoon the marmalade into 5 hot 1/2-pint canning jars, leaving 1/4 inch of space at the top, and close with the lids and rings. To process, 

In [103]:
balanced_df["full_recipe"].iloc[0]

'Recipe: Meyer Lemon Marmalade\nIngredients: lemons, sugar\nDirections: Rinse the lemons and pat dry. Halve the lemons crosswise and juice them, reserving the juice. Using a spoon, scrape the pulp and seeds from the halves. Using a sharp knife, slice the peels 1/8 inch thick., In a large, heavy saucepan, cover the strips with 8 cups of cold water and bring to a boil; boil for 1 minute. Drain the strips and rinse under cold running water. Blanch two more times; the final time, drain the strips but do not rinse them., Return the strips to the saucepan. Add the reserved juice and the sugar. Simmer over moderate heat, stirring to dissolve the sugar, then skimming any foam, until the marmalade sets, about 30 minutes., Spoon the marmalade into 5 hot 1/2-pint canning jars, leaving 1/4 inch of space at the top, and close with the lids and rings. To process, boil the jars for 15 minutes in water to cover. Let stand at room temperature for 2 days before serving., MAKE AHEAD The processed marmala

In [104]:
# GPT-2 Gewichte einfrieren, damit sie während des Fine-Tunings nicht aktualisiert werden
for param in model.parameters():
    param.requires_grad = False

In [105]:
num_classes = 9
model.out_head = torch.nn.Linear(in_features=BASE_CONFIG["emb_dim"], out_features=num_classes)

for param in model.trf_blocks[-1].parameters():
    param.requires_grad = True

for param in model.final_norm.parameters():
    param.requires_grad = True

In [106]:
inputs = tokenizer.encode("Recepie: Jewell Ball'S Chicken")
inputs = torch.tensor(inputs).unsqueeze(0)
print("Inputs:", inputs)
print("Inputs dimensions:", inputs.shape)

Inputs: tensor([[ 3041,   344, 21749,    25,  3370,   695,  6932,     6,    50, 16405]])
Inputs dimensions: torch.Size([1, 10])


In [107]:
with torch.no_grad():
    outputs = model(inputs)

print("Outputs:\n", outputs)
print("Outputs dimensions:", outputs.shape) # shape: (batch_size, num_tokens, num_classes)

Outputs:
 tensor([[[-0.9936,  1.5171, -2.0782, -1.4221, -1.4824, -1.0346,  0.6712,
          -1.2293,  0.7728],
         [-3.3238,  4.1643, -5.1299, -2.4785, -4.1144, -3.1102,  2.4958,
          -3.9160,  1.9500],
         [-3.3628,  5.1294, -5.3151, -2.8113, -6.0627, -3.1856,  3.0707,
          -4.3008,  2.6583],
         [-5.8397,  6.6605, -7.2151, -3.6975, -8.9348, -5.3379,  4.3381,
          -5.9762,  3.9989],
         [-3.3688,  5.6758, -5.7366, -3.0983, -5.7619, -3.9017,  3.5563,
          -4.7177,  2.2697],
         [-2.7462,  5.0463, -5.2150, -2.8911, -5.1549, -3.7551,  3.2650,
          -4.2068,  2.3835],
         [-3.1521,  5.4157, -5.1651, -3.5447, -5.7450, -3.8712,  3.3910,
          -4.5194,  2.4560],
         [-3.1857,  4.9747, -4.7335, -2.8445, -5.2822, -3.6373,  2.7503,
          -4.3089,  2.6566],
         [-4.0139,  6.2744, -5.0119, -3.0696, -6.1885, -4.2289,  3.5941,
          -4.8264,  2.6701],
         [-2.9842,  5.5073, -5.3107, -3.0327, -7.2403, -3.5491,  4.1003,

## Output anpassen auf Multi Label approach

In [66]:
# 1. Logits durch Sigmoid jagen und flachmachen
probas = torch.sigmoid(outputs[:, -1, :])[0].tolist()

# 2. Direkt die um +1 verschobenen IDs sammeln, die über dem Threshold liegen (1 bis 9)
active_ids = [i + 1 for i, p in enumerate(probas) if p > 0.7]

# 3. Die Genres direkt über die korrigierten IDs abfragen
active_genres = [genre_mapping[idx] for idx in active_ids]

# Reine Listen-Ausgabe
print(probas)
print(active_ids)
print(active_genres)

[0.9521650075912476, 0.9734888672828674, 0.004064538050442934, 0.21273157000541687, 0.8675329089164734, 0.9821392893791199, 0.41839268803596497, 0.13780216872692108, 0.20023545622825623]
[1, 2, 5, 6]
['bakery', 'drinks', 'fastfood', 'cereal']


In [67]:
def calc_accuracy_loader(data_loader, model, device, num_batches=None):
    model.eval()
    correct_predictions, num_examples = 0, 0

    if num_batches is None:
        num_batches = len(data_loader)
    else:
        num_batches = min(num_batches, len(data_loader))
        
    for i, (input_batch, target_batch) in enumerate(data_loader):
        if i < num_batches:
            input_batch, target_batch = input_batch.to(device), target_batch.to(device)

            with torch.no_grad():
                logits = model(input_batch)[:, -1, :]  # Logits des letzten Tokens
            
            # 1. Sigmoid anwenden und mit Threshold in 0 oder 1 umwandeln
            predicted_labels = (torch.sigmoid(logits) > 0.7).int()
            
            # 2. Prüfen, ob die Vorhersage pro Zeile EXAKT mit dem Target übereinstimmt
            # .all(dim=-1) schaut, ob alle 9 Klassen innerhalb einer Zeile korrekt sind
            correct_rows = (predicted_labels == target_batch).all(dim=-1)

            # 3. Aufsummieren
            num_examples += predicted_labels.shape[0]
            correct_predictions += correct_rows.sum().item()
        else:
            break
            
    return correct_predictions / num_examples

In [69]:
# Holt den allerersten Batch aus deinem Loader und zeigt die Form der Labels
inputs, targets = next(iter(train_loader))
print("Form des Target-Batches (Batch_Size, Klassen):", targets.shape)
print("So sieht das erste Target im Batch aus:", targets[0])

Form des Target-Batches (Batch_Size, Klassen): torch.Size([8])
So sieht das erste Target im Batch aus: tensor(7)


In [68]:
if torch.cuda.is_available():
    device = torch.device("cuda")
elif torch.backends.mps.is_available():
    # Use PyTorch 2.9 or newer for stable mps results
    major, minor = map(int, torch.__version__.split(".")[:2])
    if (major, minor) >= (2, 9):
        device = torch.device("mps")
    else:
        device = torch.device("cpu")
else:
    device = torch.device("cpu")

print("Device:", device)

model.to(device) # no assignment model = model.to(device) necessary for nn.Module classes

torch.manual_seed(123) # For reproducibility due to the shuffling in the training data loader

train_accuracy = calc_accuracy_loader(train_loader, model, device, num_batches=10)
val_accuracy = calc_accuracy_loader(val_loader, model, device, num_batches=10)
test_accuracy = calc_accuracy_loader(test_loader, model, device, num_batches=10)

print(f"Training accuracy: {train_accuracy*100:.2f}%")
print(f"Validation accuracy: {val_accuracy*100:.2f}%")
print(f"Test accuracy: {test_accuracy*100:.2f}%")

Device: cpu


RuntimeError: The size of tensor a (9) must match the size of tensor b (8) at non-singleton dimension 1

### 🔄 Pivot-Entscheidung: Übergang von Multi-Label zu Single-Label

**Ursprüngliche Überlegung:**
Es wurde evaluiert, das Modell mittels eines Multi-Label-Ansatzes (9 unabhängige Klassen via Sigmoid-Aktivierung und `BCEWithLogitsLoss`) zu trainieren. Ziel war es, Rezepten theoretisch mehrere Genres gleichzeitig zuzuweisen (z. B. *„Kuchen“* und *„Süßspeise“*).

**Problemstellung & Limitation der Daten:**
Die empirische Überprüfung des Datensatzes ($N > 2.000.000$ Zeilen) ergab, dass die Ground-Truth-Labels strikt als exklusive Single-Labels (disjunkte Klassen von 1 bis 9) strukturiert sind. Ein Training mit Sigmoid/BCE auf Single-Label-Daten führt zu einer Fehloptimierung (*False Negative Bias*): Das Modell würde aktiv dafür bestraft werden, logische Überschneidungen zu erlernen (z. B. dass ein Kuchen auch süß ist), weil die anderen Klassen im Datensatz per Definition auf `0` gesetzt sind.

**Gewählte Lösung (Architektur-Anpassung):**
Aus Gründen der mathematischen Korrektheit wird das Modell auf eine klassische **Single-Label-Klassifikation** umgestellt.
* **Aktivierung:** Softmax statt Sigmoid
* **Loss-Funktion:** `CrossEntropyLoss`
* **Multi-Label-Simulation im Deployment:** Da der Softmax-Ausgang eine Wahrscheinlichkeitsverteilung über alle Genres liefert, kann im späteren Verlauf (Inferenz) dennoch ein Schwellenwert (z. B. alle Genres $> 20\%$) genutzt werden, um dem Nutzer "weiche", mehrfache Genre-Vorschläge zu machen.

In [108]:
probas = torch.softmax(outputs[:, -1, :], dim=-1)
label = torch.argmax(probas)
predicted_idx = label.item()

# um eins verschoben, weil die IDs in genre_mapping bei 1 beginnen, die Modellvorhersage aber bei 0
final_class_id = predicted_idx + 1
genre_name = genre_mapping.get(final_class_id, "Unbekannt")
print("Probabilities:", probas[0].tolist())
print("Class label:", predicted_idx)
print(f"Echte ID (für Mapping): {final_class_id} -> {genre_name}")

Probabilities: [0.00014682230539619923, 0.7154873013496399, 1.4334793377202004e-05, 0.00013987973215989769, 2.081597585856798e-06, 8.345466630998999e-05, 0.17521308362483978, 4.556787098408677e-05, 0.1088676005601883]
Class label: 1
Echte ID (für Mapping): 2 -> drinks


In [109]:
def calc_accuracy_loader(data_loader, model, device, num_batches=None):
    model.eval()
    correct_predictions, num_examples = 0, 0

    if num_batches is None:
        num_batches = len(data_loader)
    else:
        num_batches = min(num_batches, len(data_loader))
    for i, (input_batch, target_batch) in enumerate(data_loader):
        if i < num_batches:
            input_batch, target_batch = input_batch.to(device), target_batch.to(device)

            with torch.no_grad():
                logits = model(input_batch)[:, -1, :]  # Logits of last output token
            predicted_labels = torch.argmax(logits, dim=-1)

            num_examples += predicted_labels.shape[0]
            correct_predictions += (predicted_labels == target_batch).sum().item()
        else:
            break
    return correct_predictions / num_examples

In [110]:
if torch.cuda.is_available():
    device = torch.device("cuda")
elif torch.backends.mps.is_available():
    # Use PyTorch 2.9 or newer for stable mps results
    major, minor = map(int, torch.__version__.split(".")[:2])
    if (major, minor) >= (2, 9):
        device = torch.device("mps")
    else:
        device = torch.device("cpu")
else:
    device = torch.device("cpu")

print("Device:", device)

model.to(device) # no assignment model = model.to(device) necessary for nn.Module classes

torch.manual_seed(123) # For reproducibility due to the shuffling in the training data loader

train_accuracy = calc_accuracy_loader(train_loader, model, device, num_batches=10)
val_accuracy = calc_accuracy_loader(val_loader, model, device, num_batches=10)
test_accuracy = calc_accuracy_loader(test_loader, model, device, num_batches=10)

print(f"Training accuracy: {train_accuracy*100:.2f}%")
print(f"Validation accuracy: {val_accuracy*100:.2f}%")
print(f"Test accuracy: {test_accuracy*100:.2f}%")

Device: cpu
Training accuracy: 11.25%
Validation accuracy: 18.75%
Test accuracy: 17.50%


In [112]:
def calc_loss_batch(input_batch, target_batch, model, device):
    input_batch, target_batch = input_batch.to(device), target_batch.to(device)
    logits = model(input_batch)[:, -1, :]  # Logits of last output token
    loss = torch.nn.functional.cross_entropy(logits, target_batch)
    return loss

In [ ]:
def calc_loss_loader(data_loader, model, device, num_batches=None):
    total_loss = 0.
    if len(data_loader) == 0:
        return float("nan")
    elif num_batches is None:
        num_batches = len(data_loader)
    else:
        # Reduce the number of batches to match the total number of batches in the data loader
        # if num_batches exceeds the number of batches in the data loader
        num_batches = min(num_batches, len(data_loader))
    for i, (input_batch, target_batch) in enumerate(data_loader):
        if i < num_batches:
            loss = calc_loss_batch(input_batch, target_batch, model, device)
            total_loss += loss.item()
        else:
            break
    return total_loss / num_batches

In [114]:
with torch.no_grad():
    train_loss = calc_loss_loader(train_loader, model, device, num_batches=5)
    val_loss = calc_loss_loader(val_loader, model, device, num_batches=5)
    test_loss = calc_loss_loader(test_loader, model, device, num_batches=5)

print(f"Training loss: {train_loss:.3f}")
print(f"Validation loss: {val_loss:.3f}")
print(f"Test loss: {test_loss:.3f}")

Training loss: 9.179
Validation loss: 8.536
Test loss: 5.491


## finetuning the model

In [115]:
# Overall the same as `train_model_simple` in chapter 5
def train_classifier_simple(model, train_loader, val_loader, optimizer, device, num_epochs,
                            eval_freq, eval_iter):
    # Initialize lists to track losses and examples seen
    train_losses, val_losses, train_accs, val_accs = [], [], [], []
    examples_seen, global_step = 0, -1

    # Main training loop
    for epoch in range(num_epochs):
        model.train()  # Set model to training mode

        for input_batch, target_batch in train_loader:
            optimizer.zero_grad() # Reset loss gradients from previous batch iteration
            loss = calc_loss_batch(input_batch, target_batch, model, device)
            loss.backward() # Calculate loss gradients
            optimizer.step() # Update model weights using loss gradients
            examples_seen += input_batch.shape[0] # New: track examples instead of tokens
            global_step += 1

            # Optional evaluation step
            if global_step % eval_freq == 0:
                train_loss, val_loss = evaluate_model(
                    model, train_loader, val_loader, device, eval_iter)
                train_losses.append(train_loss)
                val_losses.append(val_loss)
                print(f"Ep {epoch+1} (Step {global_step:06d}): "
                      f"Train loss {train_loss:.3f}, Val loss {val_loss:.3f}")

        # Calculate accuracy after each epoch
        train_accuracy = calc_accuracy_loader(train_loader, model, device, num_batches=eval_iter)
        val_accuracy = calc_accuracy_loader(val_loader, model, device, num_batches=eval_iter)
        print(f"Training accuracy: {train_accuracy*100:.2f}% | ", end="")
        print(f"Validation accuracy: {val_accuracy*100:.2f}%")
        train_accs.append(train_accuracy)
        val_accs.append(val_accuracy)

    return train_losses, val_losses, train_accs, val_accs, examples_seen

In [116]:
# Same as chapter 5
def evaluate_model(model, train_loader, val_loader, device, eval_iter):
    model.eval()
    with torch.no_grad():
        train_loss = calc_loss_loader(train_loader, model, device, num_batches=eval_iter)
        val_loss = calc_loss_loader(val_loader, model, device, num_batches=eval_iter)
    model.train()
    return train_loss, val_loss

In [117]:
import time

start_time = time.time()

torch.manual_seed(123)

optimizer = torch.optim.AdamW(model.parameters(), lr=5e-5, weight_decay=0.1)

num_epochs = 5
train_losses, val_losses, train_accs, val_accs, examples_seen = train_classifier_simple(
    model, train_loader, val_loader, optimizer, device,
    num_epochs=num_epochs, eval_freq=50, eval_iter=5,
)

end_time = time.time()
execution_time_minutes = (end_time - start_time) / 60
print(f"Training completed in {execution_time_minutes:.2f} minutes.")

Ep 1 (Step 000000): Train loss 8.636, Val loss 8.116


KeyboardInterrupt: 

In [ ]:
import matplotlib.pyplot as plt

def plot_values(epochs_seen, examples_seen, train_values, val_values, label="loss"):
    fig, ax1 = plt.subplots(figsize=(5, 3))

    # Plot training and validation loss against epochs
    ax1.plot(epochs_seen, train_values, label=f"Training {label}")
    ax1.plot(epochs_seen, val_values, linestyle="-.", label=f"Validation {label}")
    ax1.set_xlabel("Epochs")
    ax1.set_ylabel(label.capitalize())
    ax1.legend()

    # Create a second x-axis for examples seen
    ax2 = ax1.twiny()  # Create a second x-axis that shares the same y-axis
    ax2.plot(examples_seen, train_values, alpha=0)  # Invisible plot for aligning ticks
    ax2.set_xlabel("Examples seen")

    fig.tight_layout()  # Adjust layout to make room
    plt.savefig(f"{label}-plot.pdf")
    plt.show()

In [ ]:
epochs_tensor = torch.linspace(0, num_epochs, len(train_losses))
examples_seen_tensor = torch.linspace(0, examples_seen, len(train_losses))

plot_values(epochs_tensor, examples_seen_tensor, train_losses, val_losses)

In [ ]:
epochs_tensor = torch.linspace(0, num_epochs, len(train_accs))
examples_seen_tensor = torch.linspace(0, examples_seen, len(train_accs))

plot_values(epochs_tensor, examples_seen_tensor, train_accs, val_accs, label="accuracy")

In [ ]:
train_accuracy = calc_accuracy_loader(train_loader, model, device)
val_accuracy = calc_accuracy_loader(val_loader, model, device)
test_accuracy = calc_accuracy_loader(test_loader, model, device)

print(f"Training accuracy: {train_accuracy*100:.2f}%")
print(f"Validation accuracy: {val_accuracy*100:.2f}%")
print(f"Test accuracy: {test_accuracy*100:.2f}%")

## use the model to label recipes

In [ ]:
def classify_recipe(text, model, tokenizer, device, genre_mapping, max_length=None, pad_token_id=50256):
    model.eval()

    # 1. Inputs für das Modell vorbereiten
    input_ids = tokenizer.encode(text)
    supported_context_length = model.pos_emb.weight.shape[0]

    # Sicherstellen, dass max_length gesetzt ist
    assert max_length is not None, (
        "max_length must be specified. If you want to use the full model context, "
        "pass max_length=model.pos_emb.weight.shape[0]."
    )
    assert max_length <= supported_context_length, (
        f"max_length ({max_length}) exceeds model's supported context length ({supported_context_length})."
    )     
    
    # Text abschneiden, falls er zu lang ist
    input_ids = input_ids[:min(max_length, supported_context_length)]
    
    # Padding hinzufügen, um auf max_length zu kommen
    input_ids += [pad_token_id] * (max_length - len(input_ids))
    input_tensor = torch.tensor(input_ids, device=device).unsqueeze(0) # Batch-Dimension hinzufügen

    # 2. Modell-Inferenz (Vorhersage)
    with torch.no_grad():
        logits = model(input_tensor)[:, -1, :]  # Logits des letzten Tokens
    
    # Den Index mit der höchsten Wahrscheinlichkeit holen (0 bis 8)
    predicted_idx = torch.argmax(logits, dim=-1).item()

    # 3. Index (0-8) wieder auf die echte ID (1-9) für das Mapping umrechnen (+ 1)
    final_class_id = predicted_idx + 1
    
    # Genre-Namen aus deinem Dictionary holen
    genre_name = genre_mapping.get(final_class_id, "Unbekanntes Genre")

    return genre_name

In [ ]:
text_1 = (
    "You are a winner you have been specially"
    " selected to receive $1000 cash or a $2000 award."
)

print(classify_recipe(
    text_1, model, tokenizer, device, genre_mapping, max_length=train_dataset.max_length
))

In [ ]:
text_2 = (
    "Hey, just wanted to check if we're still on"
    " for dinner tonight? Let me know!"
)

print(classify_recipe(
    text_2, model, tokenizer, device, genre_mapping, max_length=train_dataset.max_length
))

## Save and load model

In [ ]:
torch.save(model.state_dict(), "review_classifier.pth")

In [ ]:
model_state_dict = torch.load("review_classifier.pth", map_location=device, weights_only=True)
model.load_state_dict(model_state_dict)